In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

---

# **Silver Data Transformations: Netflix_Titles**
Read the bronze `netflix_titles` table, clean and transform it, and upsert it into silver so there is exactly one row per `show_id`. Incremental loading and duplicate handling are managed by:

- **Incremental Loading (Watermark)**: The latest `_ingested_date` already in the silver table is used as the watermark, and only bronze rows with a newer `_ingested_date` are processed. On the very first run, when silver doesn't exist yet, the watermark comes from the `ingested_date` widget which is set at the orchastration layer by Azure DataFactory.

- **Cleaning Steps**
  - Drop exact duplicate rows.
  - Cast `show_id` to `int` (invalid values become NULL) and filter out rows where it is NULL.
  - Drop the `_rescued_data` column.
  - Keep only rows where `type` is `Movie` or `TV Show`, which removes malformed records.
  - Cast `duration_minutes` and `duration_seasons` to `int` and fill missing values with 0.
  - Derive `short_title` (the title text before the first `:`) and `type_flag` (1 for Movie, 2 for TV Show).

- **Deduplication**: A window over `show_id` is ordered by `_ingested_date` (newest first) and then `_file_modified` (newest first, used as a tiebreaker for rows ingested at the same time). Only the first row per `show_id` is kept, so the MERGE never sees multiple source rows for the same key.

In [0]:
dbutils.widgets.text("ingested_date", "")
_last_ingested_date = None

if spark.catalog.tableExists("netflix_catalog.silver.netflix_titles"):
    _last_ingested_date = spark.table("netflix_catalog.silver.netflix_titles") \
                               .agg(max("_ingested_date")).collect()[0][0]

# silver gave nothing then use the widget
if _last_ingested_date is None:
    _last_ingested_date = dbutils.widgets.get("ingested_date").strip() or None

print("watermark used:", _last_ingested_date)

watermark used: None


In [0]:
df = spark.read.format("delta") \
               .load("abfss://bronze@netflixprojectetedlll.dfs.core.windows.net/netflix_titles")

if _last_ingested_date is not None:
    df = df.filter(col("_ingested_date") > to_timestamp(lit(_last_ingested_date)))

In [0]:
df = df.dropDuplicates()

In [0]:
df = df.withColumn("show_id", col("show_id").try_cast("int")) \
        .drop("_rescued_data") \
        .filter(col("show_id").isNotNull()) \
        .filter(col("type").isin("Movie", "TV Show"))

In [0]:
df = df.withColumn("duration_minutes", col("duration_minutes").try_cast("int")) \
        .withColumn("duration_seasons", col("duration_seasons").try_cast("int")) \
        .fillna({"duration_minutes": 0, "duration_seasons": 0})

In [0]:
df = df.withColumn("short_title", split(col("title"), ":")[0]) \
        .withColumn("type_flag", when(col("type") == "Movie", 1).otherwise(2))

In [0]:
w = Window.partitionBy("show_id").orderBy(
    col("_ingested_date").desc(),
    col("_file_modified").desc()
)

df = df.withColumn("rn", row_number().over(w)) \
       .filter(col("rn") == 1) \
       .drop("rn")

---

# **UPSERT**

In [0]:
df.createOrReplaceTempView("silver_source")

In [0]:
if spark.catalog.tableExists("netflix_catalog.silver.netflix_titles"):
    spark.sql("""
              MERGE INTO netflix_catalog.silver.netflix_titles AS trg
              USING silver_source AS src
              ON trg.show_id = src.show_id
              WHEN MATCHED AND (src._ingested_date > trg._ingested_date) THEN UPDATE SET *
              WHEN NOT MATCHED THEN INSERT *
              """)

    print("Merged into netflix_catalog.silver.netflix_titles")

else:
    spark.sql("""
        CREATE SCHEMA IF NOT EXISTS netflix_catalog.silver
        """)
    spark.sql("""
        CREATE TABLE IF NOT EXISTS netflix_catalog.silver.netflix_titles
        USING DELTA
        LOCATION 'abfss://silver@netflixprojectetedlll.dfs.core.windows.net/netflix_titles'
        AS SELECT * FROM silver_source
    """)

    print("Created table netflix_catalog.silver.netflix_titles")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7193991352658713>, line 13
     10     print("Merged into netflix_catalog.silver.netflix_titles")
     12 else:
---> 13     spark.sql("""
     14         CREATE TABLE IF NOT EXISTS netflix_catalog.silver.netflix_titles
     15         USING DELTA
     16         LOCATION 'abfss://silver@netflixprojectetedlll.dfs.core.windows.net/netflix_titles'
     17         AS SELECT * FROM silver_source
     18     """)
     20     print("Created table netflix_catalog.silver.netflix_titles")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:919, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    916         _views.append(SubqueryAlias(df._plan, name))
    918 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 919 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    9

---

In [0]:
%sql
SELECT * FROM netflix_catalog.silver.netflix_titles

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7193991352658708>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT * FROM netflix_catalog.silver.netflix_titles\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseException as e:
    208     self.driver_ac